# UC-08 · Compare twin pumps (A vs B)

**Use case:** [UC-08](../docs/product/use-cases.md) - *two pumps built for the same
service are tested; how do they differ?*

`pump.compare.compare_curves` evaluates H, P and eta of several fitted curves at the
chosen capacities and reports the relative delta $(x - x_{ref})/x_{ref}$ against the
**first** curve (the reference).  `plot_comparison` overlays the curves.

Pump A is the P-1001A water test.  Pump B is a synthetic twin whose head is 2 % lower
and whose power is 3 % higher; its efficiency is set consistently with both.

**Acceptance:** the reported deltas equal the injected differences (-2 %, +3 % and
$0.98/1.03 - 1$).

## 1 · Setup and the two curves

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import Q_, Fluid, Water, TestPoint, PerformanceCurve

# Illustrative water test of pump P-1001A (1480 rpm):
#          Q [m3/h]  Head [m]  P [kW]  eta [%]
P1001A = [
    (160.0, 125.37, 77.97, 70.03),
    (240.0, 119.11, 95.17, 81.82),
    (320.0, 110.79, 114.44, 84.44),
    (400.0, 100.01, 147.26, 74.00),
    (480.0, 86.72, 173.98, 65.13),
    (540.0, 75.60, 194.76, 57.14),
]
water = Fluid("Test water", density=Q_(1000.0, "kg/m**3"))


def build_curve(rows, fluid, speed, degree=3, head_k=1.0, power_k=1.0, eff_k=1.0):
    "Measured rows -> PerformanceCurve with Head / eta pinned on the points."
    points = [
        TestPoint(
            fluid=fluid,
            capacity=Q_(q, "m**3/h"),
            speed_of_rotation=speed,
            _head=Q_(h * head_k, "m"),
            _efficiency=Q_(e * eff_k, "percent"),
            breaking_power=Q_(p * power_k, "kW"),
        )
        for q, h, p, e in rows
    ]
    return PerformanceCurve(fluid, points, polynomial_degree=degree)

from pump.compare import compare_curves, plot_comparison

speed = Q_(1480, "rpm")
curve_a = build_curve(P1001A, water, speed)
curve_b = build_curve(P1001A, water, speed, head_k=0.98, power_k=1.03, eff_k=0.98 / 1.03)
curves = {"P-1001A": curve_a, "P-1001B": curve_b}

## 2 · The comparison table

Capacities are Pint quantities in any unit.

In [ ]:
at = [Q_(240, "m**3/h"), Q_(320, "m**3/h"), Q_(1409, "gallon/minute")]
table = compare_curves(curves, at)
print("reference:", table.tags[0])
print(table.as_text())

## 3 · Overlay plot

In [ ]:
png = plot_comparison(curves, rated_q=Q_(320, "m**3/h"))
display(Image(data=png.getvalue()))

## 4 · A single curve cannot be compared

In [ ]:
try:
    compare_curves({"only": curve_a}, at)
    needs_two = False
except ValueError as exc:
    needs_two = True
    print("ValueError:", exc)

## 5 · Acceptance

In [ ]:
assert table.tags == ("P-1001A", "P-1001B")
assert len(table.rows) == 3
for row in table.rows:
    # the reference carries no delta; B differs by the injected factors
    assert "P-1001A" not in row.deltas
    d = row.deltas["P-1001B"]
    assert abs(d["head"] - (-0.02)) < 1e-6
    assert abs(d["power"] - 0.03) < 1e-6
    assert abs(d["efficiency"] - (0.98 / 1.03 - 1.0)) < 1e-6

# the 1409 gpm row is the 320 m3/h row in other units
assert abs(table.rows[2].q.to("m**3/h").magnitude - 320.0) < 0.2
assert needs_two
print("UC-08 acceptance OK")